In [2]:
!pip install ecdsa base58

import hashlib
import json
import base58
from ecdsa import SigningKey, VerifyingKey, SECP256k1, BadSignatureError

## a) & b) The `Wallet` class: key generation and address derivation

In [ ]:
class Wallet:
    """A minimal ECDSA wallet: private key, public key, and a derived address."""

    def __init__(self, name: str):
        self.name = name
        # Private key: a random 256-bit scalar on secp256k1
        self.private_key: SigningKey = SigningKey.generate(curve=SECP256k1)
        # Public key: the point private_key * G on the curve
        self.public_key: VerifyingKey = self.private_key.get_verifying_key()
        # Address: see derive_address() below for the documented steps
        self.address: str = self.derive_address()

    def derive_address(self) -> str:
        pubkey_bytes = b"\x04" + self.public_key.to_string()  

        sha256_1 = hashlib.sha256(pubkey_bytes).digest()
        ripemd160 = hashlib.new("ripemd160")
        ripemd160.update(sha256_1)
        pubkey_hash = ripemd160.digest()                      

        version = b"\x00"                                     
        versioned_payload = version + pubkey_hash

        checksum_full = hashlib.sha256(hashlib.sha256(versioned_payload).digest()).digest()
        checksum = checksum_full[:4]

        address_bytes = versioned_payload + checksum
        return base58.b58encode(address_bytes).decode()

    def sign(self, payload: dict) -> bytes:
        """Sign the canonical JSON encoding of a transaction payload."""
        message = self._canonical_bytes(payload)
        return self.private_key.sign(message, hashfunc=hashlib.sha256)

    @staticmethod
    def verify(public_key: VerifyingKey, payload: dict, signature: bytes) -> bool:
        """Verify a signature against a payload using the signer's public key."""
        message = Wallet._canonical_bytes(payload)
        try:
            return public_key.verify(signature, message, hashfunc=hashlib.sha256)
        except BadSignatureError:
            return False

    @staticmethod
    def _canonical_bytes(payload: dict) -> bytes:
        # sort_keys ensures a deterministic byte representation, so signer
        # and verifier always hash/sign the exact same bytes
        return json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()

    def __repr__(self):
        return f"<Wallet {self.name} address={self.address}>"

### Generate wallets for Alice and Bob

In [4]:
alice = Wallet("Alice")
bob = Wallet("Bob")

for w in (alice, bob):
    print(f"{w.name}'s wallet")
    print(f"  Private key (hex): {w.private_key.to_string().hex()}")
    print(f"  Public key  (hex): 04{w.public_key.to_string().hex()}")
    print(f"  Address          : {w.address}")
    print()

Alice's wallet
  Private key (hex): 6a2f3c3875f2e830ec37d03d40140585ca8f88d0158c0fa24b0fe1bf1f3c1169
  Public key  (hex): 04cfb952940b22d8b9bb4fcde473377c301c94d86a4dcc2234ee6c7f8202e9989ef331c07b07e268b35d412c71f5cb4682f4f1ba98fceb033e939702001aecb7ef
  Address          : 1KeJyNTpYSivCpWDtkgXNfHpsBV1qaeoAA

Bob's wallet
  Private key (hex): da1fa8ac70ffc376665f3803d8bee6fc9d85df8d90313025010d3238e6fccd62
  Public key  (hex): 044e2300fad69722ef35eb72ee761ced521dbf0b319c36d3fbd1a1565ad85d20ed5e4755f658dce8e383f0322cc029905976d1c87f5b92abc632360d8c075a3be9
  Address          : 1PWet2z53BwWF1M6RHXBKrKRs5j2xEs97w



## c) Signing, verification, and tamper detection


In [5]:
tx = {
    "from": alice.address,
    "to": bob.address,
    "amount": 50.0,
    "currency": "ZAR-remit",
    "nonce": 1,
}
print("Original transaction payload:")
print(json.dumps(tx, indent=2))

Original transaction payload:
{
  "from": "1KeJyNTpYSivCpWDtkgXNfHpsBV1qaeoAA",
  "to": "1PWet2z53BwWF1M6RHXBKrKRs5j2xEs97w",
  "amount": 50.0,
  "currency": "ZAR-remit",
  "nonce": 1
}


In [6]:
signature = alice.sign(tx)
print(f"Signature (hex, {len(signature)} bytes):\n  {signature.hex()}")

Signature (hex, 64 bytes):
  64c2e389f28617c07270ce375756f26e5550ce52dc468557b768b408d17572b4c08bd7941f81891fe475015d8db95041ed5b9999d4e756d08fd44dbf035aa6db


**Verify the signature** against the original payload, using Alice's real public key (should be `True`), and then against Bob's public key (should be `False` - proves *authenticity*: only Alice's key produces a signature that verifies under Alice's key).

In [7]:
is_valid = Wallet.verify(alice.public_key, tx, signature)
print(f"Verification with ORIGINAL payload + Alice's real public key: {is_valid}")

wrong_signer_check = Wallet.verify(bob.public_key, tx, signature)
print(f"Verification with ORIGINAL payload + Bob's public key (should be False): {wrong_signer_check}")

Verification with ORIGINAL payload + Alice's real public key: True
Verification with ORIGINAL payload + Bob's public key (should be False): False


**Tamper with the payload** and re-verify the *original* signature against the *tampered* payload - this should fail, proving *integrity*.

In [8]:
tampered_tx = dict(tx)
tampered_tx["amount"] = 5000.0   # attacker tries to inflate the transfer
print("Tampered transaction payload (amount changed):")
print(json.dumps(tampered_tx, indent=2))

is_valid_tampered = Wallet.verify(alice.public_key, tampered_tx, signature)
print(f"\nVerification of TAMPERED payload against original signature (should be False): {is_valid_tampered}")

Tampered transaction payload (amount changed):
{
  "from": "1KeJyNTpYSivCpWDtkgXNfHpsBV1qaeoAA",
  "to": "1PWet2z53BwWF1M6RHXBKrKRs5j2xEs97w",
  "amount": 5000.0,
  "currency": "ZAR-remit",
  "nonce": 1
}

Verification of TAMPERED payload against original signature (should be False): False


### Summary

In [9]:
print(f"Alice's signature verifies against the original payload : {is_valid}")
print(f"Alice's signature FAILS against Bob's public key        : {not wrong_signer_check}")
print(f"Alice's signature FAILS against the tampered payload    : {not is_valid_tampered}")
print()
print("This demonstrates the two core guarantees of a digital signature:")
print("  - Authenticity: only the holder of the private key could have produced")
print("    a signature that verifies against that key's public key.")
print("  - Integrity: any change to the signed payload, however small, causes")
print("    verification to fail.")

Alice's signature verifies against the original payload : True
Alice's signature FAILS against Bob's public key        : True
Alice's signature FAILS against the tampered payload    : True

This demonstrates the two core guarantees of a digital signature:
  - Authenticity: only the holder of the private key could have produced
    a signature that verifies against that key's public key.
  - Integrity: any change to the signed payload, however small, causes
    verification to fail.
